In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv(r'C:\Users\HP\OneDrive\Desktop\python\pyCourse\csvFiles\covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


- So here: age(numerical), gender & city(nominal-OHE), cough & has_covid(ordinal-OrinalEncoder), fever(num-SimpleImputer)

In [11]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
# train,test,split
from sklearn.model_selection import train_test_split
x_tr, x_ts, y_tr, y_ts = train_test_split(df.drop('has_covid', axis=1), df['has_covid'], test_size=0.2)

# Without ColumnTransformer

- Simple Imputer is used to transform and handle missing values

In [10]:
# adding simple imputer to fever col
si = SimpleImputer()
x_tr_fever = si.fit_transform(x_tr[['fever']])

# also for train
x_ts_fever = si.fit_transform(x_ts[['fever']])
x_tr_fever.shape

(80, 1)

In [14]:
# ordinal encoding of cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
x_tr_cough = oe.fit_transform(x_tr[['cough']])
x_ts_cough = oe.fit_transform(x_ts[['cough']])
x_tr_cough.shape

(80, 1)

In [16]:
# oneHotEncoding of gender & city
ohe = OneHotEncoder(drop='first', sparse_output=False)
x_tr_gender_city = ohe.fit_transform(x_tr[['gender','city']])
x_ts_gender_city = ohe.fit_transform(x_ts[['gender','city']])
x_tr_gender_city.shape

(80, 4)

In [19]:
# extract age 
x_tr_age = x_tr.drop(columns=['gender','fever','cough','city']).values
x_ts_age = x_tr.drop(columns=['gender','fever','cough','city']).values
x_tr_age.shape

(80, 1)

In [22]:
# now concatenate all individual col
x_tr_transformed = np.concatenate((x_tr_age,x_tr_fever,x_tr_gender_city,x_tr_cough),axis=1)
x_ts_transformed = np.concatenate((x_ts_age,x_ts_fever,x_ts_gender_city,x_ts_cough),axis=1)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 80 and the array at index 1 has size 20

# Using Column transformer

- In collntran fn, we pass two val(tranformers, remainder(drop,passthru))
- Here we pass transformers in form of tuples
- In tuples we pass: name, transfrmer opt, list of colns

In [24]:
from sklearn.compose import ColumnTransformer

In [25]:
transformer = ColumnTransformer(transformers=[
    ('trf1', SimpleImputer(), ['fever']),
    ('trf2', OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('trf3', OneHotEncoder(drop='first', sparse_output=False),['gender','city'])
], remainder='passthrough')

In [27]:
transformer.fit_transform(x_tr).shape

(80, 7)

In [29]:
transformer.fit_transform(x_ts).shape

(20, 7)

> <h4>2️⃣ ColumnTransformer </h4>
ColumnTransformer applies different preprocessing steps to different columns of a dataset simultaneously.

Key Points
1. Handles mixed data types (numeric + categorical)
2. Uses column names for clarity
3. Prevents redundant preprocessing
4. Should be used only once at pipeline start
5. Outputs NumPy array
6. Integrates perfectly with pipelines

- One-liner : ColumnTransformer defines where each preprocessing step is applied.

> <h5>✅ Thumb Rules to Use ColumnTransformer Efficiently</h5>

1. Use ColumnTransformer when data has mixed types
→ Numerical + Categorical columns.

2. Separate transformers by column type
- Numerical → Imputer + Scaler
- Categorical → Imputer + Encoder

3. Never apply OneHotEncoder to numeric columns
→ Leads to meaningless features.

4. Always use remainder='passthrough' if needed
→ Keeps untransformed columns (otherwise they are dropped).

5. ColumnTransformer should be INSIDE the Pipeline
→ Never fit it separately.

6. Use column names instead of indices (recommended)
→ Safer when dataset changes.

7. Avoid repeated preprocessing code
→ Define once inside ColumnTransformer and reuse.